Lecture 10 will cover exceptions, decorators

Reference
 * [2] Section 9.8-9.10

# Static methods

Static methods are methods that don't require the object itself.

Why are these needed?

Usually helper functions that "belong" to the class itself (i.e. are independent of actual instances)

In [29]:
from numpy.linalg import norm

class Car:
    america = True

    @staticmethod
    def kph2mph(speed):
        return speed / 1.60934

    def __init__(self, model, plate, vel):
        self.model = model
        self.velocity = vel

    def __str__(self):
        speed = norm(self.velocity)
        if self.america:
            speed = self.kph2mph(speed)
        return "Model:" + self.model + " going at " + str(speed)

In [19]:
car1 = Car("saab", "XZ123", 140)
print(car1)

Model:saab going at 86.992183131


In [21]:
Car.kph2mph(140)

86.99218313097295

# Decorators

We've seen two uses of decorators already:

```python
    @staticmethod
    def kph2mph(speed):
        return speed / 1.60934
        
    @abc.abstractmethod
    def calories(self):
        """ Returns the number of calories in the pizza """
```

In the broadest terms, decorators modify functionality. The code using `@`:
```python
@my_decorator
def my_function():
    return compute_stuff()
```
is basically the same as doing:
```python
def my_function():
    return compute_stuff()
my_function = my_decorator(my_function)
```

We can make a silly example of what it could do:

In [9]:
class always_print_hello:
    def __init__(self, f):
        self.f = f

    def __call__(self, x):
        print("Hello!")
        return self.f(x)

In [12]:
@always_print_hello
def my_function(x):
    return x+7

# Achieves the same as:
def my_function2(x):
    return x+7
my_function2 = always_print_hello(my_function2)

my_function(3) + my_function2(3)

Hello!
Hello!


20

The usefulness of decorators are perhaps not immeditaly obvious. But with some ingenuity they can be used to enrich the language itself. For example, enforcing an IntEnum to be unique:

In [13]:
import enum
@enum.unique
class GameState(enum.IntEnum):
    not_started = 0
    started = 1
    ended = 2
    paused = 2 # Opps

ValueError: duplicate values found in <enum 'GameState'>: paused -> ended

You will not be required to write any decorators on the exam, though you should know about `@staticmethod` and `@abc.abstractmethod` and how these are used.
Abstract and static methods are re-occuring themes in object oriented languages, though they might be expressed differently (i.e. using different syntax) in different languages.

Other uses of decorators are more unique "language quirks", and excessive use just makes the code hard to understand, so don't go crazy with them!

# Error handling

When dealing with user input (or even programmer input, in the example above), errors are inevitable.
We can't always know if a given piece of code will always work (i.e. cause an error on certain user inputs)

The way to deal with this is to *try* to execute segments, and deal with the errors if they occur:

In [33]:
for x in range(2,-1,-1):
    try:
        print(1.0 / x)
    except:
        print("inf")

0.5
1.0
inf


Though, we probably want to check *what* error occured, and deal with each specifically

In [32]:
for x in range(2,-1,-1):
    try:
        print(1.0 / y) # Opps, wrong variable, not defined
    except:
        print("inf") # Not correct for this error

inf
inf
inf


In [34]:
for x in range(2,-1,-1):
    try:
        print(1.0 / x) # Opps, wrong variable, not defined
    except ZeroDivisionError:
        print("inf") # Not correct for this error

0.5
1.0
inf


In [38]:
stuff = [1,2,3,4]
for x in range(8,-1,-1):
    try:
        print(stuff[x]) # Opps, wrong variable again
    except ZeroDivisionError:
        print("inf")
    except NameError:
        print("Wrong symbol!")

IndexError: list index out of range

This is why broadly catching all errors are very bad form. Don't do it!

### Using "as" to obtain info from errors

In [14]:
import sys

try:
    f = open('filename.txt')
    s = f.readline()
    i = int(s.strip())
except FileNotFoundError as e:
    print(e.errno)

2


### Finally and else

In [47]:
def divide(x, y):
    try:
        result = x / y
    except ZeroDivisionError:
        print("division by zero!")
    else:
        # Will run if we don't have an exception first:
        print("result is", result)
    finally:
        # This will always be executed last
        print("executing finally clause")

In [48]:
divide(0,0)

division by zero!
executing finally clause


In [49]:
divide(2,3)

result is 0.6666666666666666
executing finally clause


# Making your own errors

An exception is just a class:

In [15]:
class MyCustomError(Exception):
    # Inheriting from the base class for all exceptions
    pass

class NegativeValueError(ValueError):
    # Subclassing an exception to add a useful specific case
    def __init__(self, value):
        self.value = value
    def __str__(self):
        return "Value is " + repr(self.value)

Catching the exception you can get any stored values like any object:

In [81]:
def my_function(x):
    if x < 0:
        raise NegativeValueError(x)
    return x * 3

In [86]:
x = 3
try:
    my_function(x)
except NegativeValueError as e:
    print('Problem with x={}, try again.'.format(e.value))

Uncaught exceptions prints the string representations of the class:

In [83]:
my_function(-2)

NegativeValueError: Value is -2

## Possiblecustom exceptions

** Question **
* In your Poker-game library, what would be some suitable exceptions to implement?

---------------------

---------------------

---------------------
* MissingCardError
* OutOfMoneyError
* EmptyDeckError